In [120]:
import os
from dotenv import load_dotenv
import gradio as gr
import json

In [121]:
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown
from google.api_core import retry
def to_markdown(text):
    text = text.replace('•', '  *')
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [122]:
load_dotenv()
google_api_key = os.getenv('GEMINI_API_KEY')

In [123]:
genai.configure(api_key=google_api_key)

In [124]:
system_prompt ={"text":"you are an medical AI assistant, and you need to give result of reaction between compounds and medicine an it's effect on human body.You should tell wheater it is safe or not to eat the medicines together"}

In [125]:
# tool to add medicine names to a list
medicine_list = []
def medicine_tool(*medicine):
    med = medicine.lower().strip()
    if med not in medicine_list:
        medicine_list.append(med)
    return medicine_list
    

In [126]:
medicine_function={
    "name": "medicine_tool",
    "description":"gets the list of medicines.Call this whenever you want to get the list of medicines, for example, when the user asks for a list of medicines, you can call this function to get the list of medicines.",
    "parameters":{
        "type":"object",
        "properties":{
            "medicine":{
                "type":"string",
                "description":"The medicine name",
            },
        },
        "required":["medicine"],
        "additionalProperties":False
    }
}

In [127]:
tool_config = {
    "function_calling_config": {
        "mode": "ANY",
        "allowed_function_names": ["medicine_function"]
    }
}


In [128]:
user_prompt=[medicine_list]

In [129]:
# conversation=[
#     {"role": "system", "parts":[system_prompt]},
#     {"role": "user", "parts":[user_prompt]},
# ]
# try:
#     model = genai.GenerativeModel("gemini-pro",tools=[medicine_tool])
#     response = model.generate_content(conversation, tool_config=tool_config)
#     if response.prompt_feedback.block_reason:
#         print(f"Response was blocked: {response.prompt_feedback.block_reason}")
#     else:
#         print(response.text)
# except Exception as e:
#     print(f"An error occurred: {e}")


In [132]:
def message_gemini(*user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    gemini = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=system_prompt,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

In [137]:
message_gemini("paracetamol","honeytuss")

'I cannot provide specific medical advice or assess the safety of combining medications without knowing the exact formulations of "paracetamol," "honey," and "tuss" (which likely refers to a cough suppressant).  The safety depends entirely on the *specific ingredients and dosages* in each product.\n\n**Here\'s why I need more information and what the potential risks are:**\n\n* **Paracetamol (Acetaminophen):** This is a common pain reliever and fever reducer.  Overdosing on paracetamol is extremely dangerous and can cause serious liver damage.\n* **Honey:** While generally safe, honey can interact with certain medications, potentially altering their absorption or effectiveness.  It\'s also important to consider the type of honey and any added ingredients.\n* **Tuss (Cough Suppressant):** This is a broad category.  Many cough suppressants contain dextromethorphan, codeine, or other active ingredients. These can interact with acetaminophen or other drugs, potentially leading to dangerous